# Data Preprocessing Housing




#Reading the CSV file

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

df = pd.read_csv("housing.csv")
df.head(7)

In [ ]:
df.info()
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
df.describe()

#Determine the target feature

In [ ]:
target = "median_house_value"

X = df.drop(columns=target)
y = df[target]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
plt.hist(df[target], bins=30)
plt.title("Median House Value")
plt.xlabel("Median House Value")
plt.ylabel("Count")
plt.show()

In [ ]:
q1 = df[target].quantile(0.25)
q3 = df[target].quantile(0.75)
iqr = q3 - q1

low = q1 - 1.5 * iqr
high = q3 + 1.5 * iqr

outliers = df[(df[target] < low) | (df[target] > high)]

print("Lower limit:", low)
print("Upper limit:", high)
print("Number of possible outliers:", len(outliers))

## Dealing with missing values

In [ ]:
print(df.isnull().sum())

In [ ]:
df["total_bedrooms"] = df["total_bedrooms"].fillna(df["total_bedrooms"].median())

print(df.isnull().sum())

#Finding the correlation with each feature

In [ ]:
features = [
    "longitude", "latitude", "housing_median_age", "total_rooms",
    "total_bedrooms", "population", "households", "median_income"
]

correlation = df[features].corrwith(df["median_house_value"]).abs()
correlation = correlation.sort_values(ascending=False)

print(correlation)

In [ ]:
correlation.plot(kind="bar")
plt.title("Correlation with Median House Value")
plt.ylabel("Absolute correlation")
plt.show()

# Feature selection

In [ ]:
df = df.drop(columns=["population"])
df.head()

In [ ]:
print(df["ocean_proximity"].value_counts())

## Encoding categorical features

In [ ]:
encoder = OneHotEncoder(drop="first", sparse_output=False)

encoded_array = encoder.fit_transform(df[["ocean_proximity"]])
encoded_columns = encoder.get_feature_names_out(["ocean_proximity"])

encoded_df = pd.DataFrame(encoded_array, columns=encoded_columns, index=df.index)

df = df.drop(columns=["ocean_proximity"])
df = pd.concat([df, encoded_df], axis=1)

df.head()

## Train/test split

In [ ]:
X = df.drop(columns=["median_house_value"])
y = df["median_house_value"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

##  Scaling numerical features

In [ ]:
num_features = [
    "longitude", "latitude", "housing_median_age", "total_rooms",
    "total_bedrooms", "households", "median_income"
]

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_features] = scaler.fit_transform(X_train[num_features])
X_test_scaled[num_features] = scaler.transform(X_test[num_features])

X_train_scaled.head()

# Final result

In [ ]:
print("Training data:", X_train_scaled.shape)
print("Testing data:", X_test_scaled.shape)
print("Missing values in training data:", X_train_scaled.isnull().sum().sum())
print("Missing values in testing data:", X_test_scaled.isnull().sum().sum())